# 01 — Ensamblaje del Dataset (V11)

**Pipeline**: primer paso. Genera `data/dataset_v11.csv`.
**Siguiente**: `02_features_competencias.ipynb`

## Qué hace
Funde las fuentes V11 (notas 1EV, target, demografía, asistencia) en un único
DataFrame analítico y aplica los filtros de cohorte de V11.

## Dataset V11 — 1.274 alumnos-año (2010-2020)
El gate SQL ya resolvió: dedup, deterministic-final, EvaCalificacion, NoMatriculado
NULL-safe, Estado A/C, filtro de materia real. Aquí solo quedan filtros analíticos.

| Filtro V11 | Motivo |
|------------|--------|
| `total_asignaturas < 4` | Etiqueta débil: la categoría depende de 1-3 materias (59 obs) |
| Excluir Infantil / Ciclos Formativos | Calificación no comparable / masa insuficiente |

## Fuentes
| Archivo | Contenido |
|---------|-----------|
| `Q_02_v11_Final.csv` | Target (suspensos finales + categoría) |
| `Q_01_v11_Final.csv` | Notas 1EV (formato largo) |
| `Q_03_v11.csv` | Demografía (sexo, nacimiento, NEE, hermanos, repetidor) |
| `Q_04_v11.csv` | Asistencia 1EV (reciente; la vieja se suma vía Q04b) |

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Raiz TFM + import del modulo de mapeo de materias/niveles
_cwd = Path('.').resolve()
BASE_DIR = next((p for p in [_cwd, _cwd.parent, _cwd.parent.parent] if (p / 'data').exists()), _cwd.parent)
sys.path.append(str(BASE_DIR))
from src.utils.materias_map import normalizar_nivel, NIVELES_EXCLUIDOS

EXPORTS = BASE_DIR / 'data' / 'exports' / 'querys_v11'
P_TARGET = EXPORTS / 'Q_02_v11_Final.csv'
P_NOTAS  = EXPORTS / 'Q_01_v11_Final.csv'
P_DEMO   = EXPORTS / 'Q_03_v11.csv'
P_ASIST  = EXPORTS / 'Q_04_v11.csv'
P_OUT    = BASE_DIR / 'data' / 'dataset_v11.csv'
KEY = ['GuidAlumno', 'Ejercicio']

for p in [P_TARGET, P_NOTAS, P_DEMO, P_ASIST]:
    assert p.exists(), f'No encontrado: {p}'
print('Fuentes V11 encontradas OK')

Fuentes V11 encontradas OK


## 1. Target + filtro de etiqueta débil

In [2]:
# Q_02_v11_Final: target
COL_TARGET = ['GuidAlumno','Ejercicio','total_asignaturas','suspensos',
              'aprobadas','n_no_resueltas','nota_media_final','categoria_target']
target = pd.read_csv(P_TARGET, sep=';', header=None, names=COL_TARGET,
                     decimal=',', dtype={'GuidAlumno':str,'Ejercicio':int})
cat_order = ['buen_alumno','en_riesgo','con_dificultades']
target['categoria_target'] = pd.Categorical(target['categoria_target'],
                                             categories=cat_order, ordered=True)
print(f'Target crudo: {target.shape}')

# ── Filtro V11: etiqueta debil (<4 asignaturas finales) ──────────────────────
MIN_ASIG = 4
n0 = len(target)
target = target[target['total_asignaturas'] >= MIN_ASIG].copy()
print(f'Filtro total_asignaturas >= {MIN_ASIG}: {n0} -> {len(target)}  (-{n0-len(target)})')
print(target['categoria_target'].value_counts())
print('\nPor anio:'); print(target['Ejercicio'].value_counts().sort_index())

Target crudo: (1274, 8)
Filtro total_asignaturas >= 4: 1274 -> 1215  (-59)
categoria_target
buen_alumno         873
en_riesgo           216
con_dificultades    126
Name: count, dtype: int64

Por anio:
Ejercicio
2010    582
2011    301
2016      5
2018     18
2019    234
2020     75
Name: count, dtype: int64


## 2. Notas 1EV → features agregadas

In [3]:
# Q_01_v11_Final: notas 1EV (largo)
COL_NOTAS = ['GuidAlumno','Ejercicio','NivEstudio','NivCurso','IsRepetidor',
             'CodigoAsignatura','NombreAsignatura','NotaNumerico','IsAprobado','n_raw_rows']
notas = pd.read_csv(P_NOTAS, sep=';', header=None, names=COL_NOTAS,
                    decimal=',', dtype={'GuidAlumno':str,'Ejercicio':int},
                    na_values=['NULL',''])
notas['NotaNumerico'] = pd.to_numeric(notas['NotaNumerico'], errors='coerce')
notas['IsAprobado']   = pd.to_numeric(notas['IsAprobado'],   errors='coerce')
print(f'Notas 1EV (largo): {notas.shape}')
print(f'Alumnos-anio unicos: {notas[KEY].drop_duplicates().shape[0]}')

# Nivel/curso por alumno-anio (constante); NivEstudio normalizado (anios viejos sucios)
nivel_info = notas.dropna(subset=['NivEstudio']).groupby(KEY).agg(
    NivEstudio=('NivEstudio','first'), NivCurso=('NivCurso','first'),
    IsRepetidor=('IsRepetidor','first')).reset_index()
nivel_info['NivEstudio'] = nivel_info['NivEstudio'].apply(normalizar_nivel)

# Agregados numericos
notas_v = notas.dropna(subset=['NotaNumerico']).copy()
feats_notas = notas_v.groupby(KEY).agg(
    n_asignaturas_1ev=('NotaNumerico','count'),
    nota_media_1ev=('NotaNumerico','mean'),
    nota_min_1ev=('NotaNumerico','min'),
    nota_max_1ev=('NotaNumerico','max'),
    nota_std_1ev=('NotaNumerico','std'),
    n_aprobadas_1ev=('IsAprobado','sum'),
    n_suspensos_1ev=('IsAprobado', lambda x:(x==0).sum()),
).reset_index()
feats_notas['pct_aprobado_1ev'] = feats_notas['n_aprobadas_1ev']/feats_notas['n_asignaturas_1ev']
feats_notas = feats_notas.merge(nivel_info, on=KEY, how='left')
print(f'\nFeatures notas: {feats_notas.shape}')
print('Niveles normalizados:', sorted(feats_notas['NivEstudio'].dropna().unique()))

Notas 1EV (largo): (9114, 10)
Alumnos-anio unicos: 1274



Features notas: (1271, 13)
Niveles normalizados: ['Bachillerato', 'Ciclos Formativos', 'ESO', 'Primaria']


## 3. Demografía (Q03)

In [4]:
# Q_03_v11: demografia
COL_DEMO = ['GuidAlumno','Ejercicio','NivEstudio','NivCurso','Sexo',
            'NacimientoFecha','IdNacionalidad1','IsRepetidor','IdNEE','HermanosPosicion']
demo = pd.read_csv(P_DEMO, sep=';', header=None, names=COL_DEMO,
                   dtype={'GuidAlumno':str,'Ejercicio':int}, na_values=['NULL','','None'])
demo['Sexo'] = demo['Sexo'].map({'H':0,'M':1})
demo['NacimientoFecha'] = pd.to_datetime(demo['NacimientoFecha'], errors='coerce', dayfirst=False)
demo['mes_nacimiento'] = demo['NacimientoFecha'].dt.month
demo['trimestre_nacimiento'] = demo['NacimientoFecha'].dt.quarter

# Edad al inicio del curso (Ejercicio = anio de inicio). Se deja CRUDA: el centrado
# por grupo (edad relativa al curso) se hace DENTRO del pipeline, aprendiendo la media
# en TRAIN por fold (GroupCenterer) -> sin leakage. Antes se centraba aqui con la media
# global, que mezclaba train y test del mismo curso-anio.
demo['edad_inicio'] = demo['Ejercicio'] - demo['NacimientoFecha'].dt.year

# IdNacionalidad1 se descarta (cuasi-constante = Espana)
demo_feats = demo[['GuidAlumno','Ejercicio','Sexo','mes_nacimiento',
                   'trimestre_nacimiento','edad_inicio','IdNEE','HermanosPosicion']].copy()
print(f'Demografia: {demo_feats.shape}')
print(f"Cobertura Sexo: {demo_feats['Sexo'].notna().sum()}/{len(demo_feats)}")
print(demo_feats.describe().T[['mean','std','min','max']])

Demografia: (1274, 8)
Cobertura Sexo: 1274/1274


                         mean   std      min      max
Ejercicio            2012.976 4.041 2010.000 2020.000
Sexo                    0.477 0.500    0.000    1.000
mes_nacimiento          6.381 3.466    1.000   12.000
trimestre_nacimiento    2.461 1.112    1.000    4.000
edad_inicio            10.546 4.215  -15.000   44.000
IdNEE                   0.339 4.933    0.000   78.000
HermanosPosicion        0.124 0.509    0.000    4.000


## 4. Asistencia 1EV (reciente — Q04)

La asistencia de años viejos (2010-2016) se incorpora vía `Q04b` (IncIncidenciaAcumulado).

In [5]:
# Q_04_v11: asistencia reciente (IncIncidenciaSesion)
COL_ASIST = ['GuidAlumno','Ejercicio','total_incidencias_1ev',
             'no_justificadas_1ev','justificadas_1ev']
asist = pd.read_csv(P_ASIST, sep=';', header=None, names=COL_ASIST,
                    dtype={'GuidAlumno':str,'Ejercicio':int})

# Q_05 (asistencia vieja, IncIncidenciaAcumulado 2010-2016) — se une si existe
P_ASIST_VIEJA = EXPORTS / 'Q_05_v11.csv'
if P_ASIST_VIEJA.exists():
    asist_v = pd.read_csv(P_ASIST_VIEJA, sep=';', header=None, names=COL_ASIST,
                          dtype={'GuidAlumno':str,'Ejercicio':int})
    asist = pd.concat([asist, asist_v], ignore_index=True).drop_duplicates(KEY)
    print(f'Asistencia: reciente + vieja = {len(asist)} alumnos-anio')
else:
    print(f'Asistencia: solo reciente = {len(asist)} alumnos-anio  (Q04b pendiente)')
print(asist['Ejercicio'].value_counts().sort_index())

Asistencia: reciente + vieja = 376 alumnos-anio
Ejercicio
2011    153
2014      6
2016      7
2018      1
2019    198
2020     11
Name: count, dtype: int64


## 5. Merge + filtros de nivel

In [6]:
# Merge: target <- notas <- demografia <- asistencia
df = (target
      .merge(feats_notas, on=KEY, how='left', suffixes=('','_notas'))
      .merge(demo_feats,  on=KEY, how='left')
      .merge(asist, on=KEY, how='left'))

# Asistencia PONDERADA (ausencia 1x, retraso 0.5x, observacion 2x, expulsion 5x):
# los pesos son floats -> NO castear a int (truncaria 0.5, 2.5, etc.).
for c in ['total_incidencias_1ev','no_justificadas_1ev','justificadas_1ev']:
    df[c] = df[c].fillna(0).astype(float)
if 'IsRepetidor_notas' in df.columns:
    df['IsRepetidor'] = df['IsRepetidor'].fillna(df['IsRepetidor_notas'])
    df.drop(columns=['IsRepetidor_notas'], inplace=True)
print(f'Dataset mergeado: {df.shape}')

# ── Excluir Infantil / Ciclos Formativos (NivEstudio ya normalizado) ─────────
n1 = len(df)
df = df[~df['NivEstudio'].isin(NIVELES_EXCLUIDOS)].reset_index(drop=True)
print(f'Excluir {NIVELES_EXCLUIDOS}: {n1} -> {len(df)}  (-{n1-len(df)})')

# ── Excluir anios con masa insuficiente (2016 = 5 obs: ruido y EDA enganosa) ──
ANIOS_EXCLUIDOS = [2016]
n2 = len(df)
df = df[~df['Ejercicio'].isin(ANIOS_EXCLUIDOS)].reset_index(drop=True)
print(f'Excluir anios {ANIOS_EXCLUIDOS}: {n2} -> {len(df)}  (-{n2-len(df)})')
print('\nNiveles finales:'); print(df['NivEstudio'].value_counts())
print('\nPor anio:'); print(df['Ejercicio'].value_counts().sort_index())

Dataset mergeado: (1215, 28)
Excluir ['Infantil', 'Ciclos Formativos']: 1215 -> 1210  (-5)
Excluir anios [2016]: 1210 -> 1205  (-5)

Niveles finales:
NivEstudio
Primaria        774
ESO             345
Bachillerato     86
Name: count, dtype: int64

Por anio:
Ejercicio
2010    582
2011    301
2018     17
2019    234
2020     71
Name: count, dtype: int64


## 6. Guardar + verificación

In [7]:
df['target_num'] = df['categoria_target'].cat.codes
df.to_csv(P_OUT, index=False)
print(f'Guardado: {P_OUT}')
print(f'Shape final: {df.shape}')
print(f"\nTarget:\n{df['categoria_target'].value_counts()}")
miss = df.isna().sum(); miss = miss[miss>0]
print(f'\nNaN por columna:\n{miss.to_string() if len(miss) else "sin NaN"}')
df.head(4)

Guardado: C:\Users\emili\OneDrive\Escritorio\TFM\data\dataset_v11.csv
Shape final: (1205, 29)

Target:
categoria_target
buen_alumno         867
en_riesgo           213
con_dificultades    125
Name: count, dtype: int64

NaN por columna:
nota_std_1ev    39


,GuidAlumno,Ejercicio,total_asignaturas,suspensos,aprobadas,n_no_resueltas,nota_media_final,categoria_target,n_asignaturas_1ev,nota_media_1ev,nota_min_1ev,nota_max_1ev,nota_std_1ev,n_aprobadas_1ev,n_suspensos_1ev,pct_aprobado_1ev,NivEstudio,NivCurso,IsRepetidor,Sexo,mes_nacimiento,trimestre_nacimiento,edad_inicio,IdNEE,HermanosPosicion,total_incidencias_1ev,no_justificadas_1ev,justificadas_1ev,target_num
0,02A90003-455C-DF41-2B4E-4C946D30D208,2020,8,3,5,0,5.500,con_dificultades,7,7.000,6.000,9.000,1.155,7.000,0,1.000,Primaria,Cuarto de Primaria,0,1,3,1,9,0,0,0.000,0.000,0.000,2
1,02A90042-EC69-0525-78D7-4F4E230CD608,2020,8,3,5,0,5.250,con_dificultades,8,6.750,5.000,9.000,1.389,8.000,0,1.000,Primaria,Cuarto de Primaria,0,0,3,1,9,0,0,0.000,0.000,0.000,2
2,02A90003-9A90-E060-F929-C1106E30D208,2020,8,3,5,0,5.500,con_dificultades,7,7.286,6.000,9.000,0.951,7.000,0,1.000,Primaria,Cuarto de Primaria,0,0,1,1,9,0,0,0.000,0.000,0.000,2
3,02A90003-EC7A-AFC1-155F-DCF55030D208,2020,8,3,5,0,5.250,con_dificultades,7,6.857,4.000,8.000,1.345,6.000,1,0.857,Primaria,Cuarto de Primaria,0,0,1,1,9,0,0,0.000,0.000,0.000,2
